# WDL Generation Eval

This notebook evaluates how well local open-source LLMs (served via Ollama) can generate valid WDL (Workflow Description Language) scripts. It serves as a **baseline benchmark** — measuring raw model capability before any RAG augmentation or tooling.

The eval runs each test prompt at three **prompt tiers** to isolate where improvement comes from:

1. **Raw** — minimal instruction, no WDL-specific context
2. **Spec-informed** — WDL 1.0 conventions and syntax rules (what a competent author would know)
3. **Spec + example** — spec rules plus a complete WDL example (one-shot prompting)

For each tier × test case × run, the pipeline is:
`generate()` → `extract_wdl()` → `validate_wdl()` (sprocket check, 30s timeout) → results saved to `results.json`

**Prerequisites:**
- `pip install ollama`
- [Sprocket](https://github.com/stjude-rust-labs/sprocket) installed and on your `PATH`
- `ollama serve` running with a model pulled (e.g. `ollama pull llama3.1:8b`)

## 1. Imports and configuration

In [1]:
import subprocess
import tempfile
import os
import re
import json
from ollama import chat

# MODEL = "llama3.1:8b"  # also try: gemma3:4b, gemma3:12b, mistral:7b, phi4:14b
# MODEL = "gemma3:4b"  # also try: llama3.1:8b, gemma3:12b, mistral:7b, phi4:14b
# MODEL = "gemma3:12b"  # also try: llama3.1:8b, gemma3:4b, mistral:7b, phi4:14b
# MODEL = "mistral:7b"  # also try: llama3.1:8b, gemma3:4b, gemma3:12b, phi4:14b
MODEL = "phi4:14b"  # also try: llama3.1:8b, gemma3:4b, gemma3:12b, mistral:7b
N_RUNS = 2  # runs per prompt to measure consistency

# ---------------------------------------------------------------------------
# Prompt tiers — each tier adds more WDL context to the system prompt.
# This lets us measure how much of a model's performance comes from
# its training data vs. explicit specification knowledge vs. examples.
# ---------------------------------------------------------------------------

PROMPT_TIERS = {
    # Tier 1: No WDL-specific help — just "write WDL"
    "raw": (
        "You are an expert programmer. "
        "Respond only with valid code inside a ```wdl code block."
    ),

    # Tier 2: WDL 1.0 spec conventions (what a competent author would know)
    "spec": """\
You are an expert in WDL (Workflow Description Language) version 1.0.
Respond only with valid WDL code inside a ```wdl code block.

Follow these WDL 1.0 conventions exactly:

DOCUMENT STRUCTURE:
- The FIRST LINE of every WDL file MUST be `version 1.0` — even if the file only contains a single task
- Tasks define units of work; workflows orchestrate tasks

TASK STRUCTURE (every task needs ALL of these blocks):
- `meta` block — uses COLON syntax (key: "value"), NOT equals signs
- `parameter_meta` block — also uses COLON syntax (key: "value"), NOT equals signs
- `input` block with typed parameters; always include `Int cpu_cores` and `Int memory_gb` with defaults
- `command <<<` heredoc block (not command { }), starting with `set -eo pipefail`
- Use `~{variable}` interpolation inside command blocks (not ${variable})
- `output` block with typed outputs
- `runtime` block — also uses COLON syntax (docker: "image:tag", cpu: cpu_cores, memory: "~{memory_gb} GB")

IMPORTANT SYNTAX RULES:
- meta, parameter_meta, and runtime blocks all use COLON separators: `key: value`
- input and output blocks use EQUALS for assignments: `Type name = value`
- Never use `latest` for Docker tags — always pin a specific version

WORKFLOW STRUCTURE:
- Workflows call tasks with `call task_name { input: ... }`
- Use `scatter (item in collection) { ... }` for parallel execution
- Use WDL `struct` to group related inputs (e.g., sample name + files)
- Wire outputs from one task as inputs to the next: `input_name = previous_task.output_name`
- Collect workflow-level outputs in an `output` block

TYPES: String, Int, Float, Boolean, File, Array[T], Map[K,V], Pair[L,R], T? (optional)
""",

    # Tier 3: Spec conventions + a complete example (one-shot)
    "spec_plus_example": """\
You are an expert in WDL (Workflow Description Language) version 1.0.
Respond only with valid WDL code inside a ```wdl code block.

Follow these WDL 1.0 conventions exactly:

DOCUMENT STRUCTURE:
- The FIRST LINE of every WDL file MUST be `version 1.0` — even if the file only contains a single task
- Tasks define units of work; workflows orchestrate tasks

TASK STRUCTURE (every task needs ALL of these blocks):
- `meta` block — uses COLON syntax (key: "value"), NOT equals signs
- `parameter_meta` block — also uses COLON syntax (key: "value"), NOT equals signs
- `input` block with typed parameters; always include `Int cpu_cores` and `Int memory_gb` with defaults
- `command <<<` heredoc block (not command { }), starting with `set -eo pipefail`
- Use `~{variable}` interpolation inside command blocks (not ${variable})
- `output` block with typed outputs
- `runtime` block — also uses COLON syntax (docker: "image:tag", cpu: cpu_cores, memory: "~{memory_gb} GB")

IMPORTANT SYNTAX RULES:
- meta, parameter_meta, and runtime blocks all use COLON separators: `key: value`
- input and output blocks use EQUALS for assignments: `Type name = value`
- Never use `latest` for Docker tags — always pin a specific version

WORKFLOW STRUCTURE:
- Workflows call tasks with `call task_name { input: ... }`
- Use `scatter (item in collection) { ... }` for parallel execution
- Use WDL `struct` to group related inputs (e.g., sample name + files)
- Wire outputs from one task as inputs to the next: `input_name = previous_task.output_name`
- Collect workflow-level outputs in an `output` block

TYPES: String, Int, Float, Boolean, File, Array[T], Map[K,V], Pair[L,R], T? (optional)

MINIMAL EXAMPLE (a complete, valid WDL file with one task):

version 1.0

task hello {
  meta {
    description: "A simple hello world task"
    author: "Example Author"
    email: "author@example.org"
    url: "https://example.org"
    outputs: {
      greeting: "A text file with a greeting"
    }
  }

  parameter_meta {
    name: "Name to greet"
    cpu_cores: "Number of CPU cores"
    memory_gb: "Memory in GB"
  }

  input {
    String name
    Int cpu_cores = 1
    Int memory_gb = 2
  }

  command <<<
    set -eo pipefail
    echo "Hello, ~{name}!" > greeting.txt
  >>>

  output {
    File greeting = "greeting.txt"
  }

  runtime {
    docker: "ubuntu:22.04"
    cpu: cpu_cores
    memory: "~{memory_gb} GB"
  }
}
""",
}

## 2. Test cases

The evaluation dataset. Each case has an `id` (used for reporting) and a `prompt` (sent to the model). The cases are tiered by complexity, mirroring the WILDS project's module/pipeline hierarchy:

- **`single_task`** — Write one task with all required blocks (meta, parameter_meta, input, command, output, runtime). Tests whether the model knows basic WDL task anatomy.
- **`scatter_workflow`** — Workflow that scatters over samples using a struct. Tests parallel constructs and struct usage.
- **`multi_task_pipeline`** — Two tasks wired together in a workflow (output of task A feeds task B). Tests task composition and data flow.
- **`conditional_branching`** — Workflow with conditional (`if`/`else`) logic and `select_first`. Tests more advanced control flow.

These are modeled after real patterns in the WILDS WDL Library (see `ww-template`, `ww-sra-star`, etc.).

In [2]:
TEST_CASES = [
    {
        "id": "single_task",
        "prompt": (
            "Write a WDL 1.0 task called `index_bam` that takes a BAM file and "
            "runs `samtools index` on it, producing a .bai index file. "
            "The task must include all of these blocks: "
            "meta (with author, email, description, url, and outputs), "
            "parameter_meta (describing every input), "
            "input (with the BAM file, plus Int cpu_cores and Int memory_gb with defaults), "
            "command using heredoc syntax (command <<<) starting with `set -eo pipefail` "
            "and using ~{var} interpolation, "
            "output, and "
            "runtime (with a pinned Docker image tag, cpu, and memory)."
        ),
    },
    {
        "id": "scatter_workflow",
        "prompt": (
            "Write a WDL 1.0 file containing a struct called `SampleFastq` with fields "
            "`String name` and `File fastq`, a task called `run_fastqc` that runs FastQC "
            "on a single FASTQ file (with meta, parameter_meta, input, command <<<, output, "
            "and runtime blocks), and a workflow called `fastqc_pipeline` that takes an "
            "Array[SampleFastq], scatters over the samples to call `run_fastqc` on each, "
            "and collects the HTML report outputs into an Array[File]."
        ),
    },
    {
        "id": "multi_task_pipeline",
        "prompt": (
            "Write a WDL 1.0 file with two tasks and a workflow that wires them together. "
            "Task 1: `align_reads` takes paired-end FASTQ files (File r1, File r2), a "
            "reference genome File, and a sample name String, then runs `bwa mem` to produce "
            "a BAM file. "
            "Task 2: `sort_bam` takes a BAM file and runs `samtools sort` to produce a "
            "sorted BAM. "
            "Both tasks must have meta, parameter_meta, input, command <<<, output, and "
            "runtime blocks with pinned Docker images. "
            "The workflow `align_and_sort` should call align_reads, then pass its BAM output "
            "to sort_bam."
        ),
    },
    {
        "id": "conditional_branching",
        "prompt": (
            "Write a WDL 1.0 file with a task called `align_reads` that takes a File r1, "
            "an optional File? r2, a File reference, a String sample_name, and standard "
            "resource inputs. The command should run `bwa mem` with r1 only if r2 is not "
            "provided, or with both r1 and r2 if r2 is provided. "
            "Then write a workflow called `flexible_align` that takes a File r1, File? r2, "
            "and File reference as inputs. The workflow should use an `if` block: "
            "if r2 is defined, call align_reads with both files; otherwise call align_reads "
            "with only r1. Use `select_first` to pick the output BAM from whichever branch ran."
        ),
    },
]

## 3. Extract WDL from the model's response

Models usually wrap code in markdown fences (```` ```wdl ... ``` ````) even when asked not to. This helper pulls the code out of a fence if present, and falls back to the raw text otherwise. Without this, the validator would choke on the surrounding prose.

In [3]:
def extract_wdl(text: str) -> str:
    """Pull WDL out of a code fence, or return the whole thing if no fence."""
    match = re.search(r"```(?:wdl)?\n(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else text.strip()

## 4. Validate WDL with Sprocket

The grader. Writes the generated WDL to a temp file, runs `sprocket check` on it, and returns a pass/fail plus any diagnostic output. Exit code 0 means the WDL is valid; anything else is a failure.

A 30-second timeout guards against any hang (shouldn't happen with `check`, but cheap insurance). The temp file is always cleaned up via `finally`.

In [4]:
def validate_wdl(wdl_text: str) -> dict:
    """Run sprocket check. Returns pass/fail and stderr."""
    with tempfile.NamedTemporaryFile(mode="w", suffix=".wdl", delete=False) as f:
        f.write(wdl_text)
        path = f.name
    try:
        result = subprocess.run(
            ["sprocket", "check", path],
            capture_output=True, text=True, timeout=30,
        )
        return {
            "valid": result.returncode == 0,
            "stderr": (result.stderr or result.stdout).strip(),
        }
    except subprocess.TimeoutExpired:
        return {"valid": False, "stderr": "timeout"}
    finally:
        os.unlink(path)

## 5. Call the local model

Thin wrapper around Ollama's `chat` function. Accepts a `tier` parameter to select which system prompt to use — this is the key lever for measuring how much WDL-specific context helps.

In [5]:
def generate(prompt: str, tier: str = "spec_plus_example") -> str:
    """Call the local model with the specified prompt tier."""
    response = chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT_TIERS[tier]},
            {"role": "user", "content": prompt},
        ],
    )
    return response["message"]["content"]

## 6. Quick sanity check (optional)

Before running the full eval, it's worth confirming the pieces work end-to-end on one prompt. Run this cell; you should see some WDL and a `valid: True` or `False` verdict.

In [6]:
sample_tier = "spec_plus_example"
sample_raw = generate(TEST_CASES[0]["prompt"], tier=sample_tier)
sample_wdl = extract_wdl(sample_raw)
print(f"--- Generated WDL (tier: {sample_tier}) ---")
print(sample_wdl)
print("\n--- Validation ---")
print(validate_wdl(sample_wdl))

--- Generated WDL (tier: spec_plus_example) ---
version 1.0

task index_bam {
  meta {
    author: "Your Name"
    email: "your.email@example.com"
    description: "Indexes a BAM file using samtools."
    url: "https://example.org/index-bam-task"
    outputs: {
      bai_file: "A .bai index file for the input BAM file."
    }
  }

  parameter_meta {
    bam_file: "Input BAM file to be indexed."
    cpu_cores: "Number of CPU cores allocated for this task."
    memory_gb: "Amount of memory in GB allocated for this task."
  }

  input {
    File bam_file
    Int cpu_cores = 1
    Int memory_gb = 2
  }

  command <<<
    set -eo pipefail
    samtools index ~{bam_file}
  >>>

  output {
    File bai_file = "~{sub(bam_file, "\\.bam$", ".bai")}"
  }

  runtime {
    docker: "quay.io/biocontainers/samtools:1.15--h4da6232_0"
    cpu: cpu_cores
    memory: "~{memory_gb} GB"
  }
}

--- Validation ---
{'valid': True, 'stderr': ''}


## 7. The eval loop

Iterates over every prompt tier × test case × run. Results are printed as they come in and saved to `results.json` at the end.

The per-tier comparison is the headline: how much does spec knowledge and one-shot examples improve pass rates over raw generation?

In [7]:
def run_eval(tiers=None):
    """Run the eval across prompt tiers and test cases."""
    if tiers is None:
        tiers = list(PROMPT_TIERS.keys())

    all_results = []
    for tier in tiers:
        print(f"\n{'='*60}")
        print(f"TIER: {tier}")
        print(f"{'='*60}")
        tier_results = []
        for case in TEST_CASES:
            print(f"\n  --- {case['id']} ---")
            passes = 0
            runs = []
            for i in range(N_RUNS):
                raw = generate(case["prompt"], tier=tier)
                wdl = extract_wdl(raw)
                check = validate_wdl(wdl)
                runs.append({"run": i, "valid": check["valid"], "stderr": check["stderr"]})
                if check["valid"]:
                    passes += 1
                print(f"    run {i+1}: {'PASS' if check['valid'] else 'FAIL'}")
                if not check["valid"]:
                    print(f"      {check['stderr'][:200]}")
            tier_results.append({
                "id": case["id"],
                "pass_rate": passes / N_RUNS,
                "runs": runs,
            })
            print(f"    -> {passes}/{N_RUNS} passed")
        all_results.append({
            "model": MODEL,
            "tier": tier,
            "cases": tier_results,
            "avg_pass_rate": sum(r["pass_rate"] for r in tier_results) / len(tier_results),
        })

    with open("results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    # Summary table
    print(f"\n{'='*60}")
    print("SUMMARY")
    print(f"{'='*60}")
    print(f"{'Tier':<12} {'Avg Pass Rate':>14}   Per-case breakdown")
    print(f"{'-'*12} {'-'*14}   {'-'*40}")
    for tr in all_results:
        case_rates = "  ".join(f"{c['id']}:{c['pass_rate']:.0%}" for c in tr["cases"])
        print(f"{tr['tier']:<12} {tr['avg_pass_rate']:>13.0%}   {case_rates}")

    return all_results

results = run_eval(tiers=["spec_plus_example"])


TIER: spec_plus_example

  --- single_task ---
    run 1: PASS
    run 2: PASS
    -> 2/2 passed

  --- scatter_workflow ---
    run 1: PASS
    run 2: PASS
    -> 2/2 passed

  --- multi_task_pipeline ---
    run 1: PASS
    run 2: PASS
    -> 2/2 passed

  --- conditional_branching ---
    run 1: PASS
    run 2: FAIL
      error: expected `}`, but found `if` keyword
   ┌─ /var/folders/2f/kzckwczs5gd1h6h00dx3ptsc0000gn/T/tmpg4to4ap8.wdl:38:17
   │
38 │       ~{"~{r2}" if defined r2 else ""} > aligned_reads.~{sample_name}
    -> 1/2 passed

SUMMARY
Tier          Avg Pass Rate   Per-case breakdown
------------ --------------   ----------------------------------------
spec_plus_example           88%   single_task:100%  scatter_workflow:100%  multi_task_pipeline:100%  conditional_branching:50%


## 8. Inspect results

After the loop finishes, `results` holds the full record and `results.json` has the same on disk. Poke around — look at failure modes per case, dig into specific `stderr` messages, or compute whatever aggregate stats you want.

In [8]:
for tr in results:
    print(f"\n[{tr['tier']}] avg: {tr['avg_pass_rate']:.0%}")
    for c in tr["cases"]:
        print(f"  {c['id']}: {c['pass_rate']:.0%}")


[spec_plus_example] avg: 88%
  single_task: 100%
  scatter_workflow: 100%
  multi_task_pipeline: 100%
  conditional_branching: 50%


## Next steps

This notebook now benchmarks across three prompt tiers (raw → spec → spec_plus_example) to establish how much WDL-specific context helps. Future iterations:

- **RAG augmentation** — add a 4th tier that injects relevant WILDS module examples as few-shot context
- **Sprocket lint tier** — add `sprocket lint` as a stricter scoring pass (style/best practices beyond syntax)
- **Multi-model comparison** — loop over a list of `MODEL` values to compare across model families
- **Error-feedback retry** — feed sprocket errors back to the model for a "pass with retry" metric
- **Parallelization** — speed up generation with `concurrent.futures`